# Scikit learn Pipeline

- Pipelines Chains together multiple steps so that the output of each step is used as input to the next step.
- Pipelines makes it easy to apply the same preprocessing to train and test

# 1. Without Pipeline

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import MinMaxScaler

#### 1.1 Import Dataset

In [3]:
df = pd.read_csv("../../data/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
# Drop less relevant columns

df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'],axis=1,inplace=True)
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


#### 1.2 Train/Test Split

In [5]:
# Train/Test Split 
X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=['Survived']), df['Survived'], test_size=0.2, random_state=43)
print(f"Shape of Training set X : {X_train.shape}")
print(f"Shape of Test set X : {X_test.shape}")

Shape of Training set X : (712, 7)
Shape of Test set X : (179, 7)


#### 1.3 Handling Missing values 

In [6]:
# Missing Values Before: 
df.isnull().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Fare          0
Embarked      2
dtype: int64

In [7]:
# Applying Imputation

Si_age = SimpleImputer() # using mean od the data
Si_Embarked = SimpleImputer(strategy='most_frequent')

# On Training Data set
X_train_age = Si_age.fit_transform(X_train[['Age']])
X_train_embarked = Si_Embarked.fit_transform(X_train[['Embarked']])

# On Test DataSet
X_test_age = Si_age.fit_transform(X_test[['Age']])
X_test_embarked = Si_Embarked.fit_transform(X_test[['Embarked']])


#### 1.4 OneHot Encoding on 'Sex' and 'Embarked'

In [8]:
ohe_sex = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe_embarked = OneHotEncoder(sparse_output=False, handle_unknown='ignore') # No Impact on Decision tree by Multi-Collinearity

# On Train dataset
X_train_sex = ohe_sex.fit_transform(X_train[['Sex']])
X_train_embarked = ohe_embarked.fit_transform(X_train_embarked)

# On Test dataset
X_test_sex = ohe_sex.fit_transform(X_test[['Sex']])
X_test_embarked = ohe_embarked.fit_transform(X_test_embarked)



In [9]:
df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


#### 1.5 Concatenating 

In [10]:
# Extract Remaining Features
X_train_rem = X_train.drop(columns=['Sex', 'Age', 'Embarked'])
X_test_rem = X_test.drop(columns=['Sex', 'Age', 'Embarked'])

In [11]:

# Get the actual feature names from the encoders
sex_cols = ohe_sex.get_feature_names_out(['Sex']).tolist()
embarked_cols = ohe_embarked.get_feature_names_out(['Embarked']).tolist()

cols = ['Pclass', 'SibSp', 'Parch', 'Fare', 'Age'] + sex_cols + embarked_cols


X_train_transformed = pd.DataFrame(
    np.concatenate((X_train_rem, X_train_age, X_train_sex, X_train_embarked),axis=1),
    columns=cols,
    index=X_train.index
)

X_test_transformed = pd.DataFrame(
    np.concatenate((X_test_rem, X_test_age, X_test_sex, X_test_embarked),axis=1),
    columns=cols,
    index=X_test.index
)

#### 1.6 Decision Tree

In [12]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# classifier Object
clf = DecisionTreeClassifier()

# fit on train, transform both
clf.fit(X_train_transformed, y_train)

# Predict
y_pred = clf.predict(X_test_transformed)

# Accuracy of the model
print(f"Accuracy : {accuracy_score(y_test, y_pred)}")

Accuracy : 0.776536312849162


In [13]:
import pickle

pickle.dump(ohe_sex, open('../Utilities/Model/ohe_sex.pkl', 'wb'))
pickle.dump(ohe_embarked, open('../Utilities/Model/ohe_embarked.pkl', 'wb'))
pickle.dump(clf, open('../Utilities/Model/clf.pkl', 'wb'))